In [ ]:

import torch
import torch.nn.functional as F
import numpy as np

from utils.config import Config
from data_handler import DataHandler
# from classificators.dummy_classifier import DummyClassifier
# from classificators.random_forest_classifier import RandomForestClassifierSK
# from utils.utils import calculate_mcc_multilabel, plot_per_class_confusion

# pd.options.display.float_format = '{:.6f}'.format


In [35]:
# Seeding
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [36]:
config = Config()
print(config.data)
print(config.prep)

# if you use any other libraries that require seeding, set it here as well (e.g., torch.manual_seed(SEED) for PyTorch)
# -> your results should be reproducible across runs with the same seed


val_mccs = []
test_mccs = []
lr_histories_by_fold = {}

# load data
datahandler = DataHandler(config=config)



DataConfig(dataset_file='cps_data_multi_label.pkl', download_url='https://owncloud.fraunhofer.de/index.php/s/gElpu40mbgK7jau/download', sensor_cols=['Acc.x', 'Acc.y', 'Acc.z', 'Gyro.x', 'Gyro.y', 'Gyro.z', 'Baro.x'], gyro_cols=['Gyro.x', 'Gyro.y', 'Gyro.z'], test_experiment_id=1, validation_experiment_id=2, label_cols=['Driving(straight)', 'Driving(curve)', 'Lifting(raising)', 'Lifting(lowering)', 'Standing', 'Docking', 'Forks(entering or leaving front)', 'Forks(entering or leaving side)', 'Wrapping', 'Wrapping(preparation)'], superclass_mapping={'Driving(curve)': 'Driving(curve)', 'Driving(straight)': 'Driving(straight)', 'Lifting(lowering)': 'Lifting(lowering)', 'Lifting(raising)': 'Lifting(raising)', 'Wrapping': 'Turntable wrapping', 'Wrapping(preparation)': 'Stationary processes', 'Docking': 'Stationary processes', 'Forks(entering or leaving front)': 'Stationary processes', 'Forks(entering or leaving side)': 'Stationary processes', 'Standing': 'Stationary processes'})
Preprocessing

In [37]:
# =========================================================
# HELPERS
# =========================================================
def print_shapes(name, X, y=None):
    if y is None:
        print(f"{name}: {X.shape}, dtype={X.dtype}")
    else:
        print(f"{name}: X={X.shape}, y={y.shape}, X_dtype={X.dtype}, y_dtype={y.dtype}")
        

In [ ]:
def reduce_raw_to_16(X, out_points=16, chunk_size=10000):
    chunks = []
    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])
        chunk = torch.from_numpy(X[start:end]).to(device=device, dtype=torch.float32)
        chunk_reduced = F.adaptive_avg_pool1d(chunk, out_points)
        chunks.append(chunk_reduced.cpu().numpy())
        del chunk, chunk_reduced
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.concatenate(chunks, axis=0).astype(np.float16)

In [ ]:
def compute_reduced_rfft_features(X, out_bins=16, chunk_size=10000, use_log=True):
    fft_chunks = []
    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])
        chunk = torch.from_numpy(X[start:end]).to(device=device, dtype=torch.float32)

        chunk_fft = torch.fft.rfft(chunk, dim=-1)
        chunk_fft = torch.abs(chunk_fft).float()

        if use_log:
            chunk_fft = torch.log1p(chunk_fft)

        chunk_fft_reduced = F.adaptive_avg_pool1d(chunk_fft, out_bins)
        fft_chunks.append(chunk_fft_reduced.cpu().numpy())

        del chunk, chunk_fft, chunk_fft_reduced
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(fft_chunks, axis=0).astype(np.float16)

In [39]:
def merge_raw_and_fft(X_raw, X_fft):
    """
    Concatenate raw data and FFT data along channel axis.

    Input:
        X_raw shape = (N, 7, 160)
        X_fft shape = (N, 7, 160)

    Output:
        X_merged shape = (N, 14, 160)
    """
    return np.concatenate([X_raw, X_fft], axis=1).astype(np.float16)



In [ ]:
def build_14x16_features(X, chunk_size=10000):
    X_raw_16 = reduce_raw_to_16(X, out_points=16, chunk_size=chunk_size)
    X_fft_16 = compute_reduced_rfft_features(X, out_bins=16, chunk_size=chunk_size, use_log=True)
    X_merged = np.concatenate([X_raw_16, X_fft_16], axis=1).astype(np.float16)
    return X_merged

In [ ]:
def flatten_features(X):
    """
    Flatten 3D tensor into 2D for classical ML.

    Input:
        (N, 14, 160)s

    Output:
        (N, 14*160)
    """
    return X.reshape(X.shape[0], -1).astype(np.float16)



In [ ]:

# Leave-one-out: EXPERIMENT_ID = 1..4
for fold in range(1, 2):
    val_id = fold + 1 if fold < 4 else 1

    datahandler.config.data.test_experiment_id = fold
    # validation hat to be different from test
    datahandler.config.data.validation_experiment_id = val_id

    (train_x,train_y),(test_x,test_y),(val_x,val_y), target_vals = datahandler.get_data_loaders()
    print(train_x.shape, train_y.shape)
    print(test_x.shape, test_y.shape)
    print(val_x.shape, val_y.shape)
    # Ensure float32 to reduce memory
    train_x = train_x[:, :, :].astype(np.float16)
    # val_x   = val_x.astype(np.float16)
    # test_x  = test_x.astype(np.float16)

    print("\nOriginal shapes:")
    print_shapes("train", train_x, train_y)
    # print_shapes("val", val_x, val_y)
    # print_shapes("test", test_x, test_y)
    print("Target columns:", target_vals)

    # -----------------------------------------------------
    # FFT WITH SAME LENGTH
    # Input  shape: (N, 7, 160)
    # Output shape: (N, 7, 160)
    # -----------------------------------------------------
    print("\nComputing train FFT...")
    train_x_16 = build_14x16_features(train_x, chunk_size=10000)
    # val_x_16   = build_14x16_features(val_x, chunk_size=10000)
    # test_x_16  = build_14x16_features(test_x, chunk_size=10000)

    print(train_x_16.shape)  # (N, 14, 16)
    # print(val_x_16.shape)    # (N, 14, 16)
    # print(test_x_16.shape)   # (N, 14, 16)


    # -----------------------------------------------------
    # OPTIONAL FLATTENING FOR CLASSICAL ML
    # -----------------------------------------------------
    # train_x_flat = flatten_features(train_x_merged)
    # val_x_flat   = flatten_features(val_x_merged)
    # test_x_flat  = flatten_features(test_x_merged)

    # print("\nFlattened shapes:")
    # print_shapes("train_x_flat", train_x_flat, train_y)
    # print_shapes("val_x_flat", val_x_flat, val_y)
    # print_shapes("test_x_flat", test_x_flat, test_y)


Starting data preparation...
(2136440, 7, 160) (2136440, 6)
(941508, 7, 160) (941508, 6)
(1108387, 7, 160) (1108387, 6)

Original shapes:
train: X=(2136440, 7, 160), y=(2136440, 6), X_dtype=float16, y_dtype=int8
Target columns: ['Driving(curve)', 'Driving(straight)', 'Lifting(lowering)', 'Lifting(raising)', 'Stationary processes', 'Turntable wrapping']

Computing train FFT...
Processed FFT chunk: 0 to 50000 -> (50000, 7, 160)
Processed FFT chunk: 50000 to 100000 -> (50000, 7, 160)
Processed FFT chunk: 100000 to 150000 -> (50000, 7, 160)
Processed FFT chunk: 150000 to 200000 -> (50000, 7, 160)
Processed FFT chunk: 200000 to 250000 -> (50000, 7, 160)
Processed FFT chunk: 250000 to 300000 -> (50000, 7, 160)
Processed FFT chunk: 300000 to 350000 -> (50000, 7, 160)
Processed FFT chunk: 350000 to 400000 -> (50000, 7, 160)
Processed FFT chunk: 400000 to 450000 -> (50000, 7, 160)
Processed FFT chunk: 450000 to 500000 -> (50000, 7, 160)
Processed FFT chunk: 500000 to 550000 -> (50000, 7, 160)
P

In [ ]:
# The data form 
#          {
#                 acc_x:{[1,..........,160]},
#####             acc_y:{[1,..........,160]},
#####             acc_z:{[1,..........,160]},
#####             gyro_x:{[1,..........,160]},
#####             gyro_y:{[1,..........,160]},
#####             gyro_z:{[1,..........,160]},
#####             baro_x:{[1,..........,160]}
#####                                        }}
